# Analiza Efektywności Transportu i Barier Przestrzennych

Celem tego etapu jest:
1. Identyfikacja **dublujących się przystanków** (zbyt blisko siebie na tej samej linii, bez barier).
2. Wykrycie **barier architektonicznych** (duża odległość pieszego dojścia vs linia prosta).

In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# --- WCZYTANIE DANYCH ---
STOPS_PATH = "../data/processed/kielce_stops.gpkg"
GRAPH_WALK_PATH = "../data/raw/network/kielce_walk.graphml"
stops = gpd.read_file(STOPS_PATH)

# Wczytywanie grafu pieszego (to może potrwać chwilę)
G_walk = ox.load_graphml(GRAPH_WALK_PATH)
print("Graf pieszy załadowany.")

## 1. Algorytm Detekcji Dublowania Przystanków

Szukamy przystanków, które:
- Są w odległości < 200m w linii prostej.
- Ich odległość rzeczywista (sieciowa) jest zbliżona do linii prostej (brak barier).
- Obsługują te same linie (wymaga GTFS trips/stop_times).

In [ ]:
# Na razie uproszczony test dla wszystkich przystanków - szukamy par blisko siebie
from sklearn.neighbors import BallTree

# BallTree wymaga współrzędnych w radianach lub metrach. Używamy metrów (EPSG:2180)
coords = np.array(list(stops.geometry.apply(lambda x: (x.x, x.y))))
tree = BallTree(coords)

# Szukamy sąsiadów w promieniu 250m
indices, distances = tree.query_radius(coords, r=250, return_distance=True)

results = []
for i, (idx_list, dist_list) in enumerate(zip(indices, distances)):
    for j, dist in zip(idx_list, dist_list):
        if i < j: # Unikamy duplikacji par i dystansu zero
            results.append({
                'stop_a': stops.iloc[i]['stop_name'],
                'stop_b': stops.iloc[j]['stop_name'],
                'dist_straight': dist
            })

redundancy_df = pd.DataFrame(results).sort_values('dist_straight')
print(f"Znaleziono {len(redundancy_df)} par bliskich przystanków.")